# Getting Started with galpy

This tutorial introduces the most basic features of galpy: setting up gravitational potentials,
plotting rotation curves, understanding galpy's unit system, and integrating orbits.

galpy's two core modules are `galpy.potential` (gravitational potentials) and `galpy.orbit` (orbit integration).

In [ ]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

## 1. Rotation Curves

### A single disk potential

Let's start by initializing a Miyamoto-Nagai disk potential and plotting its rotation curve.
The `normalize=1.` option normalizes the potential so that the circular velocity equals 1 at R=1
(in galpy's natural units).

In [ ]:
from galpy.potential import MiyamotoNagaiPotential

mp = MiyamotoNagaiPotential(a=0.5, b=0.0375, normalize=1.0)
mp.plotRotcurve(Rrange=[0.01, 10.0], grid=1001)

### Combining multiple potentials

A realistic galaxy model requires multiple components. We can combine potentials simply by
adding them with `+`. Here we create a three-component model with a disk, a halo, and a bulge.
Note that the `normalize` values sum to 1, so the composite circular velocity is 1 at R=1.

In [ ]:
from galpy.potential import NFWPotential, HernquistPotential

mp = MiyamotoNagaiPotential(a=0.5, b=0.0375, normalize=0.6)
np_ = NFWPotential(a=4.5, normalize=0.35)
hp = HernquistPotential(a=0.6 / 8, normalize=0.05)
(hp + mp + np_).plotRotcurve(Rrange=[0.01, 10.0], grid=1001, yrange=[0.0, 1.2])

The resulting rotation curve is approximately flat, as observed in many spiral galaxies.
We can overplot the individual component curves to see how each contributes:

In [ ]:
(hp + mp + np_).plotRotcurve(Rrange=[0.01, 10.0], grid=1001, yrange=[0.0, 1.2])
mp.plotRotcurve(Rrange=[0.01, 10.0], grid=1001, overplot=True)
hp.plotRotcurve(Rrange=[0.01, 10.0], grid=1001, overplot=True)
np_.plotRotcurve(Rrange=[0.01, 10.0], grid=1001, overplot=True)

### MWPotential2014

galpy includes `MWPotential2014`, a recommended Milky-Way-like potential that is fit to
various dynamical constraints. See the Potentials section of the documentation for details.

In [ ]:
from galpy.potential import MWPotential2014

MWPotential2014.plotRotcurve(Rrange=[0.01, 10.0], grid=1001)

## 2. Natural Units

galpy works most robustly in its **natural unit system**, where:

- Positions are scaled as $x = X / [8\,\mathrm{kpc}]$
- Velocities are scaled as $v = V / [220\,\mathrm{km/s}]$

So when `normalize=1.` is set, the circular velocity is 1 at R=1, which corresponds to
220 km/s at 8 kpc for a Milky-Way-like galaxy.

The `galpy.util.conversion` module helps convert between natural and physical units.
For example, the orbital period of a circular orbit at R=1 is $2\pi$ in natural units.
In physical units:

In [ ]:
from galpy.util import conversion

print(
    "Orbital period at R=1:", 2.0 * numpy.pi * conversion.time_in_Gyr(220.0, 8.0), "Gyr"
)

About 223 Myr, as expected for a solar-like orbit.

We can also convert forces and densities. For example, the vertical force at
1.1 kpc above the plane at the solar radius in `MWPotential2014`:

In [ ]:
Fz = -MWPotential2014.zforce(1.0, 1.1 / 8.0)
print("Fz in pc/Myr^2:", Fz * conversion.force_in_pcMyr2(220.0, 8.0))
print("Fz in 2*pi*G*Msun/pc^2:", Fz * conversion.force_in_2piGmsolpc2(220.0, 8.0))

The `conversion` module also has functions for densities (`dens_in_msolpc3`),
masses (`mass_in_msol`), surface densities, and frequencies.
See the API documentation for a full list.

## 3. Physical Units with astropy

Any input to a galpy Potential, Orbit, or other object can be specified in physical units
using astropy Quantities. For example, we can set up a Miyamoto-Nagai potential with
a mass of $5 \times 10^{10}\,M_\odot$, a scale length of 3 kpc, and a scale height of 300 pc:

In [ ]:
from astropy import units

mp_phys = MiyamotoNagaiPotential(
    amp=5e10 * units.Msun, a=3.0 * units.kpc, b=300.0 * units.pc
)

When a potential is set up with physical-unit inputs, outputs are returned in physical units
by default. The circular velocity at 10 kpc:

In [ ]:
print("v_circ at 10 kpc:", mp_phys.vcirc(10.0 * units.kpc), "km/s")

If you have `astropy-units = True` in your galpy configuration file, the return value
will be an astropy Quantity with units attached. Without that setting, the value is
returned as a plain float in the default units (km/s for velocities, kpc for distances, etc.).

**Important:** If you do not specify arguments as a Quantity, galpy assumes they are in natural units.
For example, `mp_phys.vcirc(10.)` treats the input as 10 times the distance scale (80 kpc by default),
not 10 kpc.

## 4. First Orbit Integration

Let's integrate an orbit in the Miyamoto-Nagai potential. We initialize an orbit
with a five-dimensional initial condition `[R, vR, vT, z, vz]` (axisymmetric, so no azimuth):

In [ ]:
from galpy.orbit import Orbit

mp = MiyamotoNagaiPotential(a=0.5, b=0.0375, amp=1.0, normalize=1.0)
o = Orbit([1.0, 0.1, 1.1, 0.0, 0.1])
ts = numpy.linspace(0, 100, 10000)
o.integrate(ts, mp)
o.plot()

The orbit plot shows R vs. z. We can check energy conservation:

In [ ]:
o.plotE(normed=True)

The default integrator is not symplectic, so the energy error grows slowly over time,
but remains small. We can use a symplectic leapfrog integrator instead:

In [ ]:
o.integrate(ts, mp, method="leapfrog")
o.plotE(xlabel=r"$t$", ylabel=r"$E(t)/E(0)$")

With the symplectic integrator, the energy error remains bounded.

In practice, because stars have typically orbited their galaxy only tens of times,
symplectic integrators are usually unnecessary. galpy provides fast C-based integrators
accessible via the `method=` keyword, such as `method='dopr54_c'` (a high-order
Dormand-Prince method).

We can also integrate in the composite potential defined earlier:

In [ ]:
mp = MiyamotoNagaiPotential(a=0.5, b=0.0375, normalize=0.6)
np_ = NFWPotential(a=4.5, normalize=0.35)
hp = HernquistPotential(a=0.6 / 8, normalize=0.05)
o = Orbit([1.0, 0.1, 1.1, 0.0, 0.1])
ts = numpy.linspace(0, 100, 10000)
o.integrate(ts, mp + hp + np_)
o.plot()

## 5. Escape Velocity Curves

Just like rotation curves, we can plot escape velocity curves for any potential or
combination of potentials:

In [ ]:
mp_single = MiyamotoNagaiPotential(a=0.5, b=0.0375, normalize=1.0)
mp_single.plotEscapecurve(Rrange=[0.01, 10.0], grid=1001)

In [ ]:
MWPotential2014.plotEscapecurve(Rrange=[0.01, 10.0], grid=1001)

The escape velocity at the solar radius in `MWPotential2014`:

In [ ]:
v_esc = MWPotential2014.vesc(1.0)
print("Escape velocity at R=1 (natural units):", v_esc)
print("Escape velocity at solar radius:", v_esc * 220.0, "km/s")

This is approximately 513 km/s, consistent with direct measurements of the
Milky Way's escape velocity.

---

This tutorial covered the basics. For more, explore:

- **Potentials**: the full range of available potentials and their methods
- **Orbit integration**: 3D orbits, physical-unit orbits, and orbit analysis
- **Action-angle coordinates**: computing actions, angles, and frequencies
- **Distribution functions**: modeling stellar populations